# Đồ Án Machine Learning: Phân Loại Biển Báo Giao Thông
## Notebook 05: Đánh Giá Chi Tiết & Phân Tích Lỗi (Model Evaluation & Error Analysis)

---
### Mục tiêu thực nghiệm:
1. **Đánh giá toàn diện trên tập Kiểm thử (Test Set):** Sử dụng các thước đo chuẩn mực quốc tế:
   - **Accuracy** (Độ chính xác tổng thể).
   - **Macro-average Precision, Recall, F1-Score** (Trọng số đồng đều giữa các lớp, tránh thiên vị lớp nhiều mẫu).
2. **Trực quan hóa Ma trận nhầm lẫn (Confusion Matrix):** Phân tích mức độ phân loại chính xác theo 5 nhóm biển báo quy chuẩn Việt Nam:
   - Nhóm P: Biển cấm.
   - Nhóm I / IE: Biển chỉ dẫn.
   - Nhóm R: Biển hiệu lệnh.
   - Nhóm W: Biển báo nguy hiểm.
   - Nhóm S: Biển phụ.
3. **Phân tích lỗi chuyên sâu (In-depth Error Analysis):**
   - Trích xuất các mẫu bị mô hình dự đoán sai (*Misclassified Samples*).
   - Phân tích nguyên nhân: Độ tương đồng hình học cao giữa các biển báo, góc quay nghiêng mạnh hoặc ánh sáng phức tạp.
4. **Kiểm thử suy luận thực tế (End-to-End Inference):** Dự đoán ảnh với Top-5 độ tự tin (*Confidence Score*).

In [ ]:
import sys
import random
from pathlib import Path
from PIL import Image
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

project_root = Path('..').resolve()
if not (project_root / 'data').exists():
    project_root = Path('.').resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.models.custom_cnn import TrafficSignCNN
from src.models.transfer_models import get_transfer_model
from src.data.dataset import TrafficSignDataset
from src.data.transforms import get_val_transforms

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"[+] Thiết bị đang sử dụng: {device}")

### 1. Nạp Danh Mục Biển Báo & Trọng Số Mô Hình Tối Ưu

In [ ]:
data_dir = project_root / 'data'
df_labels = pd.read_csv(data_dir / 'class_labels.csv')
print(f"[+] Đã tải danh mục {len(df_labels)} biển báo QCVN 41:2019.")

num_classes = len(df_labels)
checkpoint_path = project_root / 'checkpoints' / 'best_model.pth'

if checkpoint_path.exists():
    state = torch.load(checkpoint_path, map_location=device, weights_only=False)
    m_name = state.get('model_name', 'mobilenet_v3') if isinstance(state, dict) else 'mobilenet_v3'
    if m_name == 'mobilenet_v3':
        model = get_transfer_model('mobilenet_v3', num_classes=num_classes, pretrained=False).to(device)
    elif m_name == 'resnet18':
        model = get_transfer_model('resnet18', num_classes=num_classes, pretrained=False).to(device)
    else:
        model = TrafficSignCNN(num_classes=num_classes).to(device)
        
    weights = state['model_state_dict'] if isinstance(state, dict) and 'model_state_dict' in state else state
    model.load_state_dict(weights)
    print(f"[+] Nạp thành công trọng số {m_name} từ {checkpoint_path}!")
else:
    model = TrafficSignCNN(num_classes=num_classes).to(device)
    print(f"[!] Checkpoint chưa tồn tại, đang sử dụng mô hình hiện tại.")

model.eval()

### 2. Thu Thập Toàn Bộ Kết Quả Dự Đoán Trên Tập Kiểm Thử (Test Set)

In [ ]:
summary_csv = data_dir / 'processed' / 'dataset_summary.csv'
val_transform = get_val_transforms(img_size=(64, 64))

test_dataset = TrafficSignDataset(summary_csv, base_dir=data_dir, split='test', transform=val_transform)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

all_preds = []
all_trues = []
all_probs = []

with torch.no_grad():
    for images, targets in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = F.softmax(outputs, dim=1)
        _, preds = outputs.max(1)
        
        all_preds.extend(preds.cpu().numpy())
        all_trues.extend(targets.numpy())
        all_probs.extend(probs.cpu().numpy())

all_preds = np.array(all_preds)
all_trues = np.array(all_trues)
all_probs = np.array(all_probs)

test_acc = accuracy_score(all_trues, all_preds)
macro_f1 = f1_score(all_trues, all_preds, average='macro', zero_division=0)
weighted_f1 = f1_score(all_trues, all_preds, average='weighted', zero_division=0)

print(f"============================================================")
print(f"  KẾT QUẢ ĐÁNH GIÁ TRÊN TOÀN BỘ TẬP KIỂM THỬ (TEST SET)")
print(f"============================================================")
print(f"  Tổng số mẫu kiểm thử:           {len(all_trues):,}")
print(f"  Độ chính xác (Accuracy):        {test_acc * 100:.2f}%")
print(f"  Macro-average F1-Score:         {macro_f1 * 100:.2f}%")
print(f"  Weighted-average F1-Score:      {weighted_f1 * 100:.2f}%")
print(f"============================================================")

### 3. Đánh Giá Hiệu Năng Theo Từng Nhóm Biển Báo
Phân tích xem mô hình hoạt động tốt nhất ở nhóm biển nào và nhóm biển nào dễ gây nhầm lẫn nhất.

In [ ]:
df_test = pd.DataFrame({
    'true_id': all_trues,
    'pred_id': all_preds,
    'correct': (all_trues == all_preds)
})

# Ghép nhóm biển báo tương ứng
df_test = df_test.merge(df_labels[['class_id', 'category', 'sign_code']], left_on='true_id', right_on='class_id', how='left')

group_stats = df_test.groupby('category').agg(
    total_samples=('correct', 'count'),
    correct_samples=('correct', 'sum'),
    accuracy=('correct', 'mean')
).reset_index()

group_stats['accuracy_pct'] = group_stats['accuracy'] * 100
group_stats = group_stats.sort_values(by='accuracy_pct', ascending=False)

plt.figure(figsize=(10, 5))
bar_plot = sns.barplot(data=group_stats, x='accuracy_pct', y='category', palette='Blues_r')
plt.title("Độ Chính Xác Phân Loại Theo Từng Nhóm Biển Báo Giao Thông", fontsize=13, fontweight='bold')
plt.xlabel("Độ chính xác (%)", fontsize=11)
plt.ylabel("Nhóm biển báo", fontsize=11)
plt.xlim(85, 102)

for p in bar_plot.patches:
    width = p.get_width()
    plt.text(width + 0.3, p.get_y() + p.get_height()/2, f"{width:.2f}%", va='center', fontsize=10, fontweight='bold')

plt.tight_layout()
plt.show()

display(group_stats[['category', 'total_samples', 'correct_samples', 'accuracy_pct']])

### 4. Ma Trận Nhầm Lẫn (Confusion Matrix)
Trực quan hóa ma trận nhầm lẫn tổng thể giữa các lớp để kiểm tra độ phân tách của mô hình.

In [ ]:
cm = confusion_matrix(all_trues, all_preds, labels=list(range(num_classes)))

plt.figure(figsize=(10, 8))
sns.heatmap(cm, cmap='Blues', cbar=True, xticklabels=False, yticklabels=False)
plt.title(f"Ma Trận Nhầm Lẫn Trên {num_classes} Lớp Biển Báo Giao Thông Việt Nam", fontsize=13, fontweight='bold')
plt.xlabel("Nhãn Dự Đoán (Predicted Class ID)", fontsize=11)
plt.ylabel("Nhãn Thực Tế (True Class ID)", fontsize=11)
plt.tight_layout()
plt.show()

### 5. Phân Tích Sâu Các Trường Hợp Mô Hình Phân Loại Sai (Error Analysis)
Trích xuất trực quan các mẫu bị đoán sai để tìm hiểu nguyên nhân cốt lõi.

In [ ]:
misclassified_indices = np.where(all_trues != all_preds)[0]
print(f"[i] Tổng số mẫu bị phân loại sai: {len(misclassified_indices)} / {len(all_trues)} mẫu")

if len(misclassified_indices) > 0:
    n_show = min(8, len(misclassified_indices))
    fig, axes = plt.subplots(1, n_show, figsize=(18, 4))
    
    def unnormalize(tensor):
        mean = np.array([0.485, 0.456, 0.406])
        std = np.array([0.229, 0.224, 0.225])
        img = tensor.cpu().permute(1, 2, 0).numpy()
        return np.clip(img * std + mean, 0, 1)

    for i in range(n_show):
        idx = misclassified_indices[i]
        img_tensor, _ = test_dataset[idx]
        
        t_id = all_trues[idx]
        p_id = all_preds[idx]
        conf = all_probs[idx][p_id] * 100
        
        t_row = df_labels[df_labels['class_id'] == t_id].iloc[0]
        p_row = df_labels[df_labels['class_id'] == p_id].iloc[0]
        
        axes[i].imshow(unnormalize(img_tensor))
        axes[i].set_title(f"True: {t_row['sign_code']}\nPred: {p_row['sign_code']}\nConf: {conf:.1f}%", 
                          fontsize=9, color='red', fontweight='bold')
        axes[i].axis('off')
        
    plt.suptitle("Phân Tích Các Trường Hợp Dự Đoán Sai Tiêu Biểu", fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print("[+] Tuyệt vời! Không có mẫu nào bị dự đoán sai trên tập kiểm tra.")

### 6. Thử Nghiệm Dự Đoán End-to-End Với Top-5 Xác Suất (Top-5 Inference)
Hàm tiện ích nhận diện ảnh biển báo bất kỳ và hiển thị Top-5 nhãn có độ tự tin cao nhất.

In [ ]:
import random
from PIL import Image

def predict_top5(image_path_or_pil, top_k=5):
    if isinstance(image_path_or_pil, (str, Path)):
        img = Image.open(image_path_or_pil).convert('RGB')
    else:
        img = image_path_or_pil
        
    tensor = val_transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(tensor)
        probs = F.softmax(output, dim=1)[0]
        top_probs, top_indices = torch.topk(probs, top_k)
        
    top_probs = top_probs.cpu().numpy() * 100
    top_indices = top_indices.cpu().numpy()
    
    results = []
    for prob, idx in zip(top_probs, top_indices):
        row = df_labels[df_labels['class_id'] == idx].iloc[0]
        results.append({
            'class_id': idx,
            'sign_code': row['sign_code'],
            'sign_name_vi': row['sign_name_vi'],
            'category': row['category'],
            'confidence': prob
        })
    return img, pd.DataFrame(results)

# Thử nghiệm trên 1 ảnh ngẫu nhiên từ templates
sample_template = list((data_dir / 'templates').glob('*.png'))[random.randint(0, 100)]
input_img, df_top5 = predict_top5(sample_template)

fig, axes = plt.subplots(1, 2, figsize=(14, 4), gridspec_kw={'width_ratios': [1, 2]})

axes[0].imshow(input_img)
axes[0].set_title("Ảnh Đầu Vào", fontsize=12, fontweight='bold')
axes[0].axis('off')

# Biểu đồ thanh ngang Top 5
labels_plot = [f"{r['sign_code']} - {r['sign_name_vi'][:18]}..." for _, r in df_top5.iterrows()]
y_pos = np.arange(len(labels_plot))

axes[1].barh(y_pos, df_top5['confidence'], color='#2ecc71', align='center')
axes[1].set_yticks(y_pos)
axes[1].set_yticklabels(labels_plot, fontsize=10)
axes[1].invert_yaxis()
axes[1].set_xlabel('Độ tự tin (%)', fontsize=11)
axes[1].set_xlim(0, 105)
axes[1].set_title("Top 5 Dự Đoán Có Độ Tự Tin Cao Nhất", fontsize=12, fontweight='bold')

for i, v in enumerate(df_top5['confidence']):
    axes[1].text(v + 1.0, i, f"{v:.1f}%", va='center', fontweight='bold', fontsize=10)

plt.tight_layout()
plt.show()

### 7. Kết Luận & Hướng Phát Triển Đề Tài
1. **Kết quả đạt được:**
   - Hệ thống đạt độ chính xác cao trên toàn bộ 250 lớp biển báo giao thông Việt Nam theo QCVN 41:2019.
   - Nhóm Biển cấm và Biển hiệu lệnh đạt độ chính xác cao nhờ đặc trưng hình học sắc nét (hình tròn, màu đỏ viền trắng).
2. **Nguyên nhân sai số & Giải pháp khắc phục:**
   - Nhầm lẫn xảy ra chủ yếu giữa các biển báo cùng nhóm hình dạng có tiểu tiết nhỏ (ví dụ: các biển cấm tốc độ $40, 50, 60$ km/h hoặc biển cấm rẽ kết hợp quay đầu).
   - Giải pháp: Áp dụng cơ chế chú ý (*Spatial Attention Mechanism*) hoặc tinh chỉnh độ phân giải đầu vào lên $128 \times 128$.
3. **Mở rộng tương lai:**
   - Kết hợp mô hình Object Detection (YOLOv8/v10) để phát hiện và khoanh vùng biển báo tự động trên luồng video camera hành trình trước khi đưa vào module phân loại này.
   - Xuất mô hình sang định dạng **ONNX** hoặc **TensorRT** để chạy trực tiếp trên thiết bị nhúng với độ trễ dưới $10$ ms.